In [ ]:
import asyncio
asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())

In [265]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

In [266]:
file_path = 'final_dataset.csv'
dataset = pd.read_csv(file_path)

In [267]:
print("Dataset Info:")
dataset.info()
print("\nFirst 5 Rows of the Dataset:")
print(dataset.head())

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1985 entries, 0 to 1984
Data columns (total 15 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Life expectancy                  1985 non-null   float64
 1   Adult Mortality                  1985 non-null   float64
 2   Alcohol                          1985 non-null   float64
 3   percentage expenditure           1985 non-null   float64
 4   BMI                              1985 non-null   float64
 5   Polio                            1985 non-null   float64
 6   Diphtheria                       1985 non-null   float64
 7   HIV/AIDS                         1985 non-null   float64
 8   GDP                              1985 non-null   float64
 9   thinness 10-19 years             1985 non-null   float64
 10  Income composition of resources  1985 non-null   float64
 11  Schooling                        1985 non-null   float64
 12  Econom

In [268]:
# Очистите названия столбцов
# Удалите начальные и конечные пробелы в названиях столбцов
dataset.columns = dataset.columns.str.strip()

In [269]:
# Проверьте, нет ли пропущенных и бесконечных значений
print("\nChecking for Missing Values:")
missing_values = dataset.isnull().sum()
print(missing_values)

print("\nChecking for Infinite Values:")
infinite_values = dataset.select_dtypes(include=['float64']).apply(lambda x: np.isinf(x).sum())
print(infinite_values)


Checking for Missing Values:
Life expectancy                    0
Adult Mortality                    0
Alcohol                            0
percentage expenditure             0
BMI                                0
Polio                              0
Diphtheria                         0
HIV/AIDS                           0
GDP                                0
thinness 10-19 years               0
Income composition of resources    0
Schooling                          0
Economic_health_index              0
Health_expenditure_ratio           0
Mortality_rate_ratio               0
dtype: int64

Checking for Infinite Values:
Life expectancy                      0
Adult Mortality                      0
Alcohol                              0
percentage expenditure               0
BMI                                  0
Polio                                0
Diphtheria                           0
HIV/AIDS                             0
GDP                                  0
thinness 10-19 years

In [270]:
dataset.replace([np.inf, -np.inf], np.nan, inplace=True)

dataset.fillna(dataset.median(numeric_only=True), inplace=True)


In [271]:
X = dataset.drop(columns=['Life expectancy'])
y = dataset['Life expectancy']

In [272]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [273]:
# Экспорт X_train
X_train.to_csv('X_train.csv', index=False)
print("X_train exported to 'X_train.csv'")

# Экспорт X_test
X_test.to_csv('X_test.csv', index=False)
print("X_test exported to 'X_test.csv'")

# Экспорт y_train
y_train.to_csv('y_train.csv', index=False, header=True)
print("y_train exported to 'y_train.csv'")

# Экспорт y_test
y_test.to_csv('y_test.csv', index=False, header=True)
print("y_test exported to 'y_test.csv'")


X_train exported to 'X_train.csv'
X_test exported to 'X_test.csv'
y_train exported to 'y_train.csv'
y_test exported to 'y_test.csv'


In [274]:
# Шаг 9: Определите числовые и категориальные столбцы.
numeric_features = X.select_dtypes(include=['float64']).columns
categorical_features = X.select_dtypes(include=['object']).columns

In [275]:
# Шаг 10: Создайте конвейеры предварительной обработки
# Предварительная обработка для числовых объектов
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# Предварительная обработка для категориальных объектов
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Объединить препроцессоры в столбчатый трансформатор
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

In [276]:
# Шаг 11: Создайте полный конвейер
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42))
])

In [277]:
print(model_pipeline)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  Index(['Adult Mortality', 'Alcohol', 'percentage expenditure', 'BMI', 'Polio',
       'Diphtheria', 'HIV/AIDS', 'GDP', 'thinness 10-19 years',
       'Income composition of resources', 'Schooling', 'Economic_health_index',
       'Health_expenditure_ratio', 'Mortality_rate_ratio'],
      dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index([], dtype='object'))])),
                ('model', RandomForestRegressor(random_state=42